# 🤝 第7周-Day4：多Agent协作模式

> 从单打独斗到团队协作：主Agent + 子Agent、管道 vs 并行
> 核心概念：Orchestrator-Worker、任务分发与聚合、上下文传递

**实验目标：**
1. 模拟管道（Pipeline）vs 并行（Parallel）两种多 Agent 协作模式
2. 对比两种模式的延迟和吞吐量
3. 可视化任务执行时间线

In [ ]:
# 配置 matplotlib 中文显示
from matplotlib import font_manager
import matplotlib.pyplot as plt
import numpy as np

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print(f"中文字体配置完成: {font_name}")

## 实验1：模拟管道模式 vs 并行模式

In [ ]:
import time
import random

def simulate_pipeline(tasks, stages):
    """管道模式：每个任务依次通过所有阶段"""
    results = []
    for task in tasks:
        latency = sum(random.gauss(s, 0.1*s) for s in stages)
        results.append(latency)
    return results

def simulate_parallel(tasks, stages, num_workers=3):
    """并行模式：多个任务可以同时执行"""
    # 模拟：将任务分给多个 worker，每个 worker 串行处理自己的任务
    buckets = [[] for _ in range(num_workers)]
    for i, t in enumerate(tasks):
        buckets[i % num_workers].append(t)
    results = []
    for bucket in buckets:
        for task in bucket:
            latency = sum(random.gauss(s, 0.1*s) for s in stages)
            results.append(latency)
    return results

# 参数
num_tasks = 12
stages = [1.0, 1.5, 2.0, 1.0]  # 4个处理阶段
tasks = range(num_tasks)

pipe_times = simulate_pipeline(tasks, stages)
parallel_times = simulate_parallel(tasks, stages, num_workers=3)

print(f"管道模式 - 平均延迟: {np.mean(pipe_times):.2f}s")
print(f"并行模式 - 平均延迟: {np.mean(parallel_times):.2f}s")
print(f"并行模式 - 总吞吐时间: {max(pipe_times)/len(pipe_times)*len(parallel_times)*0.4:.2f}s (模拟)")

## 实验2：任务执行时间线可视化

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6))

# 管道模式时间线
ax = axes[0]
start = 0
for i, t in enumerate(pipe_times):
    ax.barh(i, t, left=start, color="#4ECDC4", height=0.6)
    start += t
    start -= t * 0.7  # 管道重叠模拟
ax.set_title("管道模式 - 任务执行时间线")
ax.set_xlabel("时间 (s)")
ax.set_ylabel("任务编号")
ax.set_yticks(range(num_tasks))

# 并行模式时间线
ax = axes[1]
colors_p = ["#FF6B6B", "#4ECDC4", "#45B7D1"]
for i, t in enumerate(parallel_times):
    worker = i % 3
    y_base = worker * 4 + (i // 3)
    ax.barh(y_base, t, left=i * 0.5, color=colors_p[worker], height=0.6)
ax.set_title("并行模式 - 任务执行时间线 (3 Workers)")
ax.set_xlabel("时间 (s)")
ax.set_ylabel("任务/Worker")
plt.tight_layout()
plt.show()

## 实验3：Worker 数量 vs 加速比

In [ ]:
workers = [1, 2, 3, 4, 6, 8, 12]
speedups = []
for w in workers:
    pipe_total = np.mean([sum(stages) for _ in range(100)])
    par_times = simulate_parallel(range(100), stages, num_workers=w)
    # 近似并行总时间
    par_total = sum(sorted(par_times, reverse=True)[:w]) * (100 // w + 1) / (100 // w + 1)
    speedups.append(pipe_total / (np.mean(par_times) * 0.4))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(workers, speedups, 'o-', color="#2196F3", linewidth=2, markersize=8)
ax.axhline(y=speedups[0], color='gray', linestyle='--', label="串行基准")
ax.plot(workers, workers, '--', color="#FF9800", label="理想线性加速")
ax.set_xlabel("Worker 数量")
ax.set_ylabel("加速比")
ax.set_title("多Agent并行：Worker数量 vs 加速比（模拟）")
ax.legend()
plt.tight_layout()
plt.show()